In [0]:
%run ./LLM/OpenAIClient

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ac59936d-686f-466d-abbe-353c19be56e1
    Can't uninstall 'typing_extensions'. No files were found to uninstall.



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 101.0 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [0]:
%run 
./LLM/AzureAISearchClient

In [0]:
import pandas as pd

In [0]:
# 整形済みデータの読み込み
df = spark.table("poc2.plasticated_0902abe_b").toPandas()
# display(df)
print(len(df))

1654


In [0]:
df['plain_content'].describe()
df['content_len'] = df['plain_content'].apply(len)
print('-----------------')
df['content_len'].describe()

-----------------


count     1654.000000
mean       724.741838
std       1055.579204
min          0.000000
25%        316.000000
50%        432.000000
75%        655.750000
max      12636.000000
Name: content_len, dtype: float64

In [0]:
# ID列を付与
df['ID'] = range(1, len(df.index) + 1)
print(df)

                title majorcategory  ... graphdesc    ID
0        NSM001_においとは          ノウハウ  ...        []     1
1        NSM002_メカニズム          ノウハウ  ...        []     2
2     NSM003_発生シーンマップ          ノウハウ  ...        []     3
3        NSM004_材料として          ノウハウ  ...        []     4
4        NSM005_車両として          ノウハウ  ...        []     5
...               ...           ...  ...       ...   ...
1649    材料リマインダ(Z008)         リマインダ  ...        []  1650
1650    材料リマインダ(Z009)         リマインダ  ...        []  1651
1651    材料リマインダ(Z010)         リマインダ  ...        []  1652
1652    材料リマインダ(Z011)         リマインダ  ...        []  1653
1653    材料リマインダ(Z012)         リマインダ  ...        []  1654

[1654 rows x 11 columns]


In [0]:
# 抽出型のインデックス定義
index_schema = {
    "fields": [
        {"name": "ID", "type": "Edm.String", "key": True, "searchable": False},

        {"name": "Title", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "MajorCategory", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "SubCategory", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "Content", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "ContentVector", "type": "Collection(Edm.Single)", "searchable": True,"retrievable":False,"stored":False,"vectorSearchProfile": "vector-profile-1","dimensions":3072},
        {"name": "Summary", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "SummaryVector", "type": "Collection(Edm.Single)", "searchable": True,"retrievable":False,"stored":False,"vectorSearchProfile": "vector-profile-1","dimensions":3072},
        {"name": "TableDesc", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
    ],
  "semantic": {
    "defaultConfiguration": None,
    "configurations": [
      {
        "name": "q2-semantic_configuration",
        "prioritizedFields": {
          "titleField": {
            "fieldName": "Title"
          },
          "prioritizedContentFields": [
            {
              "fieldName": "Summary",
            }
          ],
        }
      }
    ]
  },
    "vectorSearch": {
        "algorithms": [
            {
                "name": "poc2-hnsw-1",
                "kind": "hnsw",
                "hnswParameters":
                {
                    "m": 4,
                    "efConstruction": 400,
                    "efSearch": 500,
                    "metric": "cosine"
                }
            },
        ],
        "profiles": [      
            {
                "name": "vector-profile-1",
                "algorithm": "poc2-hnsw-1"
            }
      ]
    },
}

In [0]:
# index作成
import traceback
index_name='index-q2poc-extractive-summary-verify'

#Azure AI Searchの情報を設定
service_name = 'ai-search-tmc-allgenai-poc'
api_version = "2024-05-01-preview"
api_key = 'xQ07KHXJ2Kms79xiw434BUWcSvik4nxLW3qlHg9IXuAzSeA31ZoX'
#AzureAISearchClientをインスタンス化
client = AzureAISearchClient(service_name, api_version, api_key)

#インデックスの削除
try:
    client.delete_index(index_name=index_name)
#インデックスの作成
    res = client.create_index(index_name=index_name, index_schema=index_schema)
    print(f"create index response:{res}")
except Exception as e:
    print(e)
    traceback.print_stack()


#インデックスの一覧を取得
indexes = client.get_indexes()
print(indexes)
print("作成完了")

Not found index-q2poc-extractive-summary-verify
create index response:{'@odata.context': 'https://ai-search-tmc-allgenai-poc.search.windows.net/$metadata#indexes/$entity', '@odata.etag': '"0x8DCD22336689DB9"', 'name': 'index-q2poc-extractive-summary-verify', 'defaultScoringProfile': None, 'fields': [{'name': 'ID', 'type': 'Edm.String', 'searchable': False, 'filterable': True, 'retrievable': True, 'stored': True, 'sortable': True, 'facetable': True, 'key': True, 'indexAnalyzer': None, 'searchAnalyzer': None, 'analyzer': None, 'normalizer': None, 'dimensions': None, 'vectorSearchProfile': None, 'vectorEncoding': None, 'synonymMaps': []}, {'name': 'Title', 'type': 'Edm.String', 'searchable': True, 'filterable': True, 'retrievable': True, 'stored': True, 'sortable': True, 'facetable': True, 'key': False, 'indexAnalyzer': None, 'searchAnalyzer': None, 'analyzer': 'ja.microsoft', 'normalizer': None, 'dimensions': None, 'vectorSearchProfile': None, 'vectorEncoding': None, 'synonymMaps': []}, 

In [0]:
# ベクトル化の動作確認
embed_model = "text-embedding-3-large"

# インスタンス化
aoai = AzureOpenAIClient()

# ベクトル化
text = "こんちちは"#df.loc[:,"report"][0]
vector = aoai.get_embedding(embed_model, text)

# 内容と次元数確認
print("次元数：", len(vector))


次元数： 3072


In [0]:
# Azure AI Searchにデータをアップロード
# 50件で1分ぐらい 890件で23分 1654件で40分
# カテゴリー、サブカテゴリー、タイトルのベクター化をやめた 1654件で18分
embed_model = "text-embedding-3-large"

#add documents
#Batch your uploads to Azure Search
#インデックスを作成するバッチ処理
def batch_upload_json_data_to_index(json_data, embed_model, batch_size=100):#batch_size=1000だと書き込み容量制限に引っかかる
    batch_array = []
    count = 0
    batch_counter = 0
    json_data = eval(json_data)
    for index, i in enumerate(json_data):
        #print(i)
        count += 1

        print(f"{index} {str(i['title'])} {len(i['plain_content'])} {str(i['plain_content'])[:30]}")
        batch_array.append(
            {
                "@search.action": "upload",
                "ID": str(i["ID"]),
                "Title": str(i["title"]),
                "MajorCategory": str(i["majorcategory"]),
                "SubCategory": str(i["subcategory"]),
                "Content": str(i["plain_content"]),
                "ContentVector": aoai.get_embedding(embed_model, str(i["plain_content"])[:8191]),
                "Summary": str(i["summary"]),
                "SummaryVector": aoai.get_embedding(embed_model, str(i["summary"])),
                "TableDesc": "",   
            }
        )

        # limit batches to 1000 records.
        # When the counter hits a number divisible by 1000, the batch is sent.
        if count % batch_size == 0:
            json = client.add_documents(index_name=index_name, documents=batch_array)
            result_list = json['value']
            for i,r in enumerate(result_list):
                if r['statusCode'] >=400:
                    import pprint
                    print(f"batch:{i}")
                    pprint.pprint(r)
            batch_counter += 1
            print(f"Batch sent! - #{batch_counter}")
            batch_array = []
        #break# 本番では削除する
    # This will catch any records left over, when not divisible by 1000
    if len(batch_array) > 0:
        json = client.add_documents(index_name=index_name, documents=batch_array)
        
        result_list = json['value']
        for i,r in enumerate(result_list):
            if r['statusCode'] >=400:
                import pprint
                print(f"final batch:{i}")
                pprint.pprint(r)
        batch_counter += 1
        print(f"Final batch sent! - #{batch_counter}")
    print("Done!")

#upload the data
json_data = df.to_json(orient='records')

batch_upload_json_data_to_index(json_data, embed_model)


0 NSM001_においとは 192 ーエアコンの「におい」とはー
エアコンのon\/オフ時に、エ
1 NSM002_メカニズム 340 ーエアコン臭のメカニズムー
1.湿り臭、オフ臭、凍結臭
エバ
2 NSM003_発生シーンマップ 919 ーエアコン臭の発生シーンマップー
イングリッシュ
エアコンの
3 NSM004_材料として 322 1材料としての基準の満足1
→イングリッシュ
車室内及び車室
4 NSM005_車両として 524 イングリッシュ
イングリッシュ
イングリッシュ
車両としては
5 NSM006_エバ表面処理改善 243 ーエバポレーター表面処理改善1
イングリッシュ
エバポレータ
6 NSM007_OFF臭、凍結臭の頻度低減 541 –オフ臭、凍結臭の頻度低減1
イングリッシュ
エアコン臭の低
7 NSM008_こもりの低減 298 イングリッシュ
ここでは、こもり臭を低減する為のエアコン制御
8 NSM009_目標範囲 338 1目標範囲1
イングリッシュ
におい強度及び発生頻度の目標範
9 NEL001_TPEとは 587 –TPEとはー
イングリッシュ
Iドヒcldlinermop
10 NEL002_TPEの分類 287 ーTPEの分類ー
イングリッシュ
熱可塑性エラストマther
11 NEL003_TPOとは? 949 –TPOとはー
オレフィン系熱可塑性エラストマーthermo
12 NEL004_構造 223 TPOの1種である「ミラストマー®」(三井化学(株)、動的架
13 NEL005_比重 167 1比重が小さいー
»イングリッシュ
TPOの1種である「ミラ
14 NEL006_耐熱性 382 TPOの1種である「ミラストマー®」(三井化学(株)、動的架
15 NEL007_耐寒性 139 1耐寒性に優れるー
イングリッシュ
TPOの1種である「ミラ
16 NEL008_熱老化性 239 1熱老化性に優れるー
TPOの1種である「ミラストマー®」(
17 NEL009_耐候性 390 1耐候性に優れるー
イングリッシュ
TPOの1種である「ミラ
18 NEL010_耐オゾン性 150 ー耐オゾン性に優れるー
イングリッシュ
TPOの1種である「
19 NEL011_電気絶縁性 120 1電気絶縁性に優れるーイングリッシュ